# PyTorch, Part 1: Classification with No Hidden Layer

CSCI 6379 · Topic 13. First contact with PyTorch: the binary and multiclass classifiers of Topics 10-12, reproduced in a few lines. Input straight to output — no hidden layer, no neural network yet.

## Binary classification (the Topic 10/22 problem)

The 3-feature dataset from Topic 10, labels 0/1. `nn.Linear(3,1)` is exactly w·x + b; `BCEWithLogitsLoss` bundles the sigmoid + log-loss; `SGD` is our gradient descent. Three input nodes, one output node — no hidden layer.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

X = torch.tensor([[1.5, 2.7, 1.3], [2.4, 1.7, 2.1], [2.5, 1.3, 2.2],
                  [8.5, 5.3, 4.8], [4.9, 6.4, 5.7], [7.2, 7.1, 7.4]])
y = torch.tensor([[1.], [1.], [1.], [0.], [0.], [0.]])

torch.manual_seed(0)
model = nn.Linear(3, 1)                        # 3 inputs -> 1 output : no hidden layer
criterion = nn.BCEWithLogitsLoss()             # sigmoid + log-loss, bundled
optimizer = optim.SGD(model.parameters(), lr=0.1)

for epoch in range(2000):
    optimizer.zero_grad()                      # clear old gradients
    loss = criterion(model(X), y)              # forward + measure error
    loss.backward()                            # autograd computes gradients
    optimizer.step()                           # gradient-descent update

test_X = torch.tensor([[2.4, 2.5, 0.7], [5.9, 4.4, 5.2],
                       [0.2, 0.5, 0.6], [4.3, 4.5, 5.5]])
test_y = torch.tensor([1., 0., 1., 0.])
probs = torch.sigmoid(model(test_X)).detach().flatten()
print("test probabilities:", probs.round(decimals=3).tolist())
print("accuracy:", ((probs >= 0.5).float() == test_y).float().mean().item())


## Multiclass on the Iris dataset, still no hidden layer

150 flowers, 4 features, 3 species (the dataset of Topic 12's assignment). Only three things change: `nn.Linear(4, 3)` (one score per class), `CrossEntropyLoss` (softmax + cross-entropy; labels are plain class indices, NOT one-hot), and `argmax` to predict.


In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)              # 150 flowers, 4 features, 3 species
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=True, random_state=42)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train)                # class indices 0/1/2 — no one-hot!
X_test  = torch.tensor(X_test,  dtype=torch.float32)
y_test  = torch.tensor(y_test)

torch.manual_seed(0)
model = nn.Linear(4, 3)                        # 4 features -> 3 class scores
criterion = nn.CrossEntropyLoss()              # softmax + cross-entropy, bundled
optimizer = optim.SGD(model.parameters(), lr=0.05)

for epoch in range(2000):                      # the loop is IDENTICAL
    optimizer.zero_grad()
    loss = criterion(model(X_train), y_train)
    loss.backward()
    optimizer.step()

pred = model(X_test).argmax(1)                 # largest score wins
print("test accuracy :", (pred == y_test).float().mean().item())
print("train accuracy:", (model(X_train).argmax(1) == y_train).float().mean().item())


## Peek inside

The model really is just a weight matrix and a bias — the same W (4x3) and b (3) we trained by hand in Topic 12.

In [ ]:
print("W (3x4):", model.weight.detach().round(decimals=3))
print("b (3):  ", model.bias.detach().round(decimals=3))
print("softmax of first 3 test flowers (rows sum to 1):")
print(torch.softmax(model(X_test[:3]), dim=1).detach().round(decimals=3))
